# ECON4305 Task 6 — Shared Data Preprocessing

Prepare one consistent dataset for the group's classification models. Run all cells from top to bottom.
Only the supplied `6_SP500 Index Dataset.xlsx` is used. The workbook is never modified.

**Forecast convention:** row `date = t` predicts whether the close at t exceeds the close at the previous observed trading session. Every predictor ends in `_lag1` and uses information through the previous observed session (`feature_date`). No same-day OHLCV or same-day technical indicator is included in X.

The source already contains 129 technical indicators. We retain them, recompute/check the 18 price/volume moving averages, and add 19 explicitly defined historical features. Other supplied indicators keep their source definitions; their original calculation code is not available, so their internal causality cannot be fully audited here.

**Team choices, not mandatory course settings:** chronological 70% train / 15% validation / remaining 15% test, one-session feature lag, and complete-case filtering after feature construction. The original leading indicator warm-up rows are not filled with invented values. No scaling, PCA, feature selection, sampling or model fitting is performed on the full dataset.


## 1. Settings and dependencies
Requires Python 3.9+, `pandas`, `numpy`, `openpyxl`. If needed, install these in your notebook environment with `%pip install pandas numpy openpyxl`.
Place the Excel next to this notebook (or in `input/`), or set `INPUT_PATH` explicitly. The Desktop fallback is for the original project machine. Outputs go to `processed/` under the current notebook working directory.


In [1]:
from pathlib import Path
import hashlib
import json
import re
import platform
import numpy as np
import pandas as pd

FILE_NAME = "6_SP500 Index Dataset.xlsx"
INPUT_PATH = None  # Example: Path("/your/path/6_SP500 Index Dataset.xlsx")
OUTPUT_DIR = Path.cwd() / "processed"
TRAIN_FRACTION = 0.70
VALID_FRACTION = 0.15
SCHEMA_VERSION = "1.0.0"
TARGET = "daily_move"
MA_WINDOWS = (5, 10, 20, 50, 100, 200)

if INPUT_PATH is None:
    candidates = [Path.cwd()/FILE_NAME, Path.cwd()/"input"/FILE_NAME,
                  Path.home()/"Desktop"/FILE_NAME]
    INPUT_PATH = next((p for p in candidates if p.is_file()), None)
if INPUT_PATH is None or not Path(INPUT_PATH).is_file():
    raise FileNotFoundError("Place the original Excel beside this notebook or set INPUT_PATH.")
INPUT_PATH = Path(INPUT_PATH)
assert 0 < TRAIN_FRACTION < 1 and 0 < VALID_FRACTION < 1
assert TRAIN_FRACTION + VALID_FRACTION < 1
print("Input:", INPUT_PATH.name)
print("Output folder:", OUTPUT_DIR.name)


FileNotFoundError: Place the original Excel beside this notebook or set INPUT_PATH.

## 2. Read the data and the supplied variable definitions
`Data` contains observations; `Sheet1` is the data dictionary. Column names are converted to lowercase snake_case. The original label name `Daily_move` becomes `daily_move`.


In [ ]:
def normalize_name(name):
    name = re.sub(r"[^a-zA-Z0-9]+", "_", str(name).strip()).strip("_").lower()
    if not name:
        raise ValueError("An input column has no usable name.")
    return name

with pd.ExcelFile(INPUT_PATH, engine="openpyxl") as workbook:
    raw = pd.read_excel(workbook, sheet_name="Data")
    definitions_raw = pd.read_excel(workbook, sheet_name="Sheet1", header=None)

column_mapping = {str(c): normalize_name(c) for c in raw.columns}
if len(set(column_mapping.values())) != len(raw.columns):
    raise ValueError("Column names collide after normalization; inspect the source headers.")
source_definitions = {}
for key, description in definitions_raw.iloc[:, :2].itertuples(index=False, name=None):
    if pd.notna(key) and pd.notna(description):
        source_definitions[normalize_name(key)] = str(description).strip()

print("Source shape:", raw.shape)
print("Source date range:", raw["Date"].min(), "to", raw["Date"].max())
print(raw.iloc[:3, :7].to_string(index=False))


Source shape: (9243, 136)
Source date range: 1985-12-31 00:00:00 to 2022-08-31 00:00:00
      Date  Daily_move       Open       High        Low      Close    Volume
1985-12-31         NaN 210.679993 211.610001 210.679993 211.279999 112700000
1986-01-02         0.0 211.279999 211.279999 208.929993 209.589996  98960000
1986-01-03         1.0 209.589996 210.880005 209.509995 210.880005 105000000


## 3. Validate and clean
Remove fully empty rows and identical duplicate records, then sort by date. Conflicting duplicate dates, invalid dates, non-finite/missing OHLCV, non-positive prices, negative volume or inconsistent OHLC ranges stop execution for review; they are not silently repaired.

Numeric conversion failures and infinite indicator values become missing and are logged. The target is checked against close-to-close movement. A missing target is reconstructed only when both adjacent observed closes are known. The first observation remains unlabelled. Valid extreme market moves are retained. We do not insert weekends/holidays, interpolate prices or winsorize against full-sample statistics.


In [ ]:
def clean_source(source):
    log = []
    def record(step, count, detail):
        log.append({"step": step, "count": int(count), "detail": detail})
    d = source.rename(columns={c: normalize_name(c) for c in source.columns}).copy()
    if d.columns.duplicated().any():
        raise ValueError("Duplicate normalized column names.")
    required = {"date", TARGET, "open", "high", "low", "close", "volume"}
    if not required.issubset(d.columns):
        raise ValueError(f"Missing required columns: {sorted(required-set(d.columns))}")
    record("input_rows", len(d), "Rows read from Data sheet")
    empty = d.isna().all(axis=1)
    record("empty_rows_removed", empty.sum(), "Only fully empty rows")
    d = d.loc[~empty].copy()
    # This supplied workbook stores actual Excel dates, not numeric serial strings.
    if pd.api.types.is_numeric_dtype(d["date"]):
        raise ValueError("Numeric dates require an explicitly verified date origin.")
    d["date"] = pd.to_datetime(d["date"], errors="coerce")
    if d["date"].isna().any():
        raise ValueError("Missing/unparseable dates; inspect the original data.")
    if not d["date"].eq(d["date"].dt.normalize()).all():
        raise ValueError("Expected daily dates without intraday timestamps.")
    for col in d.columns.drop("date"):
        before = d[col].notna()
        converted = pd.to_numeric(d[col], errors="coerce")
        bad = before & converted.isna()
        inf = np.isinf(converted)
        if bad.any():
            record("numeric_parse_to_missing", bad.sum(), col)
        if inf.any():
            record("infinity_to_missing", inf.sum(), col)
        d[col] = converted.replace([np.inf, -np.inf], np.nan)
    duplicate = d.duplicated()
    record("identical_duplicates_removed", duplicate.sum(), "All columns must agree")
    d = d.loc[~duplicate].sort_values("date", kind="stable").reset_index(drop=True)
    if d["date"].duplicated().any():
        raise ValueError("Conflicting records on the same date; no arbitrary row selected.")
    core = ["open", "high", "low", "close", "volume"]
    if d[core].isna().any().any():
        raise ValueError("OHLCV contains missing/non-numeric/infinite values; review source.")
    if (d[["open", "high", "low", "close"]] <= 0).any().any() or (d.volume < 0).any():
        raise ValueError("Non-positive price or negative volume.")
    invalid_range = (d.high < d[["open", "close", "low"]].max(axis=1)) | \
                    (d.low > d[["open", "close", "high"]].min(axis=1))
    if invalid_range.any():
        raise ValueError("Inconsistent OHLC high/low range.")
    observed = d[TARGET].notna()
    if not d.loc[observed, TARGET].isin([0, 1]).all():
        raise ValueError("Target must be 0, 1 or missing.")
    previous_close = d.close.shift(1)
    expected = (d.close > previous_close).astype(float).where(previous_close.notna())
    mismatch = observed & expected.notna() & d[TARGET].ne(expected)
    if mismatch.any():
        raise ValueError(f"{int(mismatch.sum())} supplied targets disagree with adjacent closes.")
    fillable = d[TARGET].isna() & expected.notna()
    record("missing_targets_reconstructed", fillable.sum(), "From observed adjacent closes only")
    d.loc[fillable, TARGET] = expected[fillable]
    d.loc[expected.isna(), TARGET] = np.nan
    record("validated_rows", len(d), "Chronological, unique dates; original sessions retained")
    return d, log

clean, cleaning_log = clean_source(raw)
input_missing = clean.isna().sum()
print("Validated rows:", len(clean))
print("Columns with missing values:", int(input_missing.gt(0).sum()))
print("Missing indicator values after the first 250 observations:",
      int(clean.drop(columns=["date", TARGET]).iloc[250:].isna().sum().sum()))


Validated rows: 9243
Columns with missing values: 122
Missing indicator values after the first 250 observations: 0


## 4. Recalculate common indicators and add historical features
All calculations below are trailing, never centred. Windows count observed trading sessions.

- Recalculate the supplied price SMA/EMA and volume SMA for 5, 10, 20, 50, 100 and 200 sessions. EMA uses `adjust=False`, alpha = 2/(window+1), seeded by the first close; early values are hidden until the full window exists. Compare against the supplied values and report differences.
- Add 1/5/10/20-session simple returns; one-session log return; 20/60-session sample standard deviation of daily returns (unannualized, ddof=1).
- Add high-low range / close, close / open - 1, and open / previous close - 1.
- Add close / SMA - 1 for the six windows, volume / 20-session volume SMA, MACD / close and ATR / close. Returns/ratios are decimals, not percentage points.

Existing indicators such as RSI, ATR, MACD and OBV retain their provided values. Their units are documented in the source dictionary; do not assume all original indicators are decimals.


In [ ]:
def build_features(d):
    source_cols = [c for c in d.columns if c not in ("date", TARGET)]
    f = d[source_cols].copy()
    descriptions = {c: source_definitions.get(c, "Provided numeric source field") for c in source_cols}
    origins = {c: "source" for c in source_cols}
    checks = []
    for w in MA_WINDOWS:
        computed = {
            f"p_sma_{w}": d.close.rolling(w, min_periods=w).mean(),
            f"p_ema_{w}": d.close.ewm(span=w, adjust=False, min_periods=w).mean(),
            f"v_sma_{w}": d.volume.rolling(w, min_periods=w).mean(),
        }
        for name, values in computed.items():
            if name in f:
                valid = f[name].notna() & values.notna()
                checks.append({"indicator": name, "rows_compared": int(valid.sum()),
                               "max_absolute_difference": float((f.loc[valid, name]-values[valid]).abs().max()),
                               "matches_with_tolerance": bool(np.allclose(f.loc[valid, name], values[valid], rtol=1e-7, atol=1e-8))})
            f[name] = values
            origins[name] = "recomputed_moving_average"
    def add(name, values, description):
        if name in f:
            raise ValueError(f"New feature collides with source field: {name}")
        f[name] = values
        origins[name] = "derived"
        descriptions[name] = description
    for w in (1, 5, 10, 20):
        add(f"return_{w}d", d.close / d.close.shift(w) - 1,
            f"Close / close {w} observed sessions earlier - 1; decimal")
    add("log_return_1d", np.log(d.close / d.close.shift(1)), "Natural log of close / previous close")
    for w in (20, 60):
        add(f"volatility_{w}d", f.return_1d.rolling(w, min_periods=w).std(ddof=1),
            f"Trailing {w}-session sample standard deviation of daily simple returns; not annualized")
    add("intraday_range_ratio", (d.high-d.low)/d.close, "(High - low) / close")
    add("intraday_return", d.close/d.open-1, "Close / open - 1; decimal")
    add("opening_gap_return", d.open/d.close.shift(1)-1, "Open / previous session close - 1; decimal")
    for w in MA_WINDOWS:
        add(f"close_sma_{w}_ratio", d.close/f[f"p_sma_{w}"]-1,
            f"Close / trailing {w}-session price SMA - 1")
    add("volume_sma_20_ratio", d.volume/f.v_sma_20.replace(0, np.nan),
        "Volume / trailing 20-session volume SMA; undefined if denominator is zero")
    add("macd_close_ratio", f.macd/d.close, "Supplied MACD / close")
    add("atr_close_ratio", f.atr/d.close, "Supplied ATR / close")
    return f.replace([np.inf, -np.inf], np.nan), descriptions, origins, checks

features_at_close, feature_definitions, feature_origins, indicator_checks = build_features(clean)
print(pd.DataFrame(indicator_checks).to_string(index=False))
print("Features before lagging:", len(features_at_close.columns))
print("Added features:", sum(v == "derived" for v in feature_origins.values()))


indicator  rows_compared  max_absolute_difference  matches_with_tolerance
  p_sma_5           9239             7.958079e-12                    True
  p_ema_5           9239             9.322321e-12                    True
  v_sma_5           9239             0.000000e+00                    True
 p_sma_10           9234             7.958079e-12                    True
 p_ema_10           9234             1.000444e-11                    True
 v_sma_10           9234             0.000000e+00                    True
 p_sma_20           9224             8.185452e-12                    True
 p_ema_20           9224             1.091394e-11                    True
 v_sma_20           9224             0.000000e+00                    True
 p_sma_50           9194             7.275958e-12                    True
 p_ema_50           9194             1.091394e-11                    True
 v_sma_50           9194             0.000000e+00                    True
p_sma_100           9144             9

## 5. Align features and target; remove unavailable rows
Calculate on the intact time series **before** dropping incomplete rows. This preserves the meaning of a one-session lag. Then shift every predictor by one observation, retaining the original target date.

Rows with unavailable features/labels are excluded and every excluded date is recorded. For the supplied file this removes the leading 251 rows (250-session ROC warm-up plus one feature lag), rather than median/zero-filling them. No additional lag should be applied by group members.


In [ ]:
def align_model_table(d, contemporaneous_features):
    lagged = contemporaneous_features.shift(1).add_suffix("_lag1")
    table = pd.concat([d[["date"]], d["date"].shift(1).rename("feature_date"),
                       lagged, d[TARGET]], axis=1)
    eligible = table.notna().all(axis=1)
    excluded = table.loc[~eligible, ["date", "feature_date"]].copy()
    excluded["missing_feature_count"] = lagged.loc[~eligible].isna().sum(axis=1)
    excluded["missing_target"] = table.loc[~eligible, TARGET].isna()
    excluded["missing_feature_date"] = table.loc[~eligible, "feature_date"].isna()
    ready = table.loc[eligible].reset_index(drop=True)
    ready[TARGET] = ready[TARGET].astype("int8")
    return ready, excluded

model_data, excluded_rows = align_model_table(clean, features_at_close)
FEATURE_COLUMNS = [c + "_lag1" for c in features_at_close.columns]
first_eligible_date = model_data.date.min() if len(model_data) else None
interior_exclusions = (excluded_rows.date > first_eligible_date).sum() if first_eligible_date is not None else 0
cleaning_log += [
    {"step": "incomplete_rows_excluded", "count": len(excluded_rows),
     "detail": "Missing lagged features, target or feature_date; see excluded_rows.csv"},
    {"step": "interior_incomplete_rows", "count": int(interior_exclusions),
     "detail": "Excluded dates after first usable observation; inspect if non-zero"},
    {"step": "model_ready_rows", "count": len(model_data), "detail": "Complete, one-session-lagged predictors"},
]
if len(model_data) < 100:
    raise ValueError("Too few complete observations; review missing data before changing policy.")
print("Excluded rows:", len(excluded_rows), "| Interior exclusions:", int(interior_exclusions))
print("Model-ready shape:", model_data.shape)
print("Target dates:", model_data.date.min().date(), "to", model_data.date.max().date())


Excluded rows: 251 | Interior exclusions: 0
Model-ready shape: (8992, 156)
Target dates: 1986-12-29 to 2022-08-31


## 6. Freeze a common chronological split
The split is a team default, not a teacher-mandated ratio. All models must use the same dates and feature list. No shuffling. Validation is for tuning; test is reserved for final evaluation. For cross-validation use expanding/rolling time splits. Scaling, PCA, imputation, feature selection and any resampling must be fitted separately inside training folds, not here.

This represents successive one-session forecasts: a validation/test row can use observed market information from the previous session, even if that previous session is inside validation/test. This is not a fixed-origin forecast of the entire test period.


In [ ]:
n = len(model_data)
train_end = int(n * TRAIN_FRACTION)
valid_end = train_end + int(n * VALID_FRACTION)
model_data["split"] = np.where(np.arange(n) < train_end, "train",
                              np.where(np.arange(n) < valid_end, "validation", "test"))
model_data = model_data[["date", "feature_date", "split"] + FEATURE_COLUMNS + [TARGET]]
split_summary = []
for name in ("train", "validation", "test"):
    part = model_data.loc[model_data.split.eq(name)]
    split_summary.append({"split": name, "rows": len(part),
                          "start": str(part.date.min().date()), "end": str(part.date.max().date()),
                          "up_count": int(part[TARGET].sum()),
                          "down_or_flat_count": int(part[TARGET].eq(0).sum())})
print(pd.DataFrame(split_summary).to_string(index=False))


     split  rows      start        end  up_count  down_or_flat_count
     train  6294 1986-12-29 2011-12-09      3373                2921
validation  1348 2011-12-12 2017-04-21       720                 628
      test  1350 2017-04-24 2022-08-31       748                 602


## 7. Validate the shared interface
Check labels, finite predictors, date order and strict one-session alignment. Independently verify a return and a moving average against source prices. A prefix test verifies that removing later observations does not change earlier calculated features. This tests this notebook's calculations, not the inaccessible original implementations of every supplied indicator.


In [ ]:
assert model_data.date.is_unique and model_data.date.is_monotonic_increasing
assert model_data.feature_date.lt(model_data.date).all()
assert model_data[TARGET].isin([0, 1]).all()
assert not model_data.isna().any().any()
assert np.isfinite(model_data[FEATURE_COLUMNS].to_numpy(dtype=float)).all()
assert all(c.endswith("_lag1") for c in FEATURE_COLUMNS)
assert TARGET not in FEATURE_COLUMNS and "date" not in FEATURE_COLUMNS
assert model_data.loc[model_data.split.eq("train"), "date"].max() < model_data.loc[model_data.split.eq("validation"), "date"].min()
assert model_data.loc[model_data.split.eq("validation"), "date"].max() < model_data.loc[model_data.split.eq("test"), "date"].min()

positions = clean.set_index("date").index.get_indexer(model_data.date)
expected_lagged = features_at_close.iloc[positions-1].to_numpy(dtype=float)
np.testing.assert_allclose(model_data[FEATURE_COLUMNS].to_numpy(dtype=float), expected_lagged)
np.testing.assert_array_equal(model_data.feature_date.to_numpy(), clean.date.iloc[positions-1].to_numpy())
expected_labels = (clean.close.iloc[positions].to_numpy() > clean.close.iloc[positions-1].to_numpy()).astype(int)
np.testing.assert_array_equal(model_data[TARGET].to_numpy(), expected_labels)

i = 500
np.testing.assert_allclose(features_at_close.loc[i, "p_sma_20"], np.mean(clean.close.iloc[i-19:i+1]))
np.testing.assert_allclose(features_at_close.loc[i, "return_5d"], clean.close.iloc[i]/clean.close.iloc[i-5]-1)
prefix_features, _, _, _ = build_features(clean.iloc[:600].copy())
pd.testing.assert_frame_equal(prefix_features, features_at_close.iloc[:600])
print("PASS: schema, labels, chronological splits, one-session alignment and calculation checks.")


PASS: schema, labels, chronological splits, one-session alignment and calculation checks.


## 8. Export the data and interface
`sp500_model_ready.csv` is the shared training input. Use only the ordered predictors in `feature_columns.json`. Metadata columns (`date`, `feature_date`, `split`) and target are not predictors. `data_dictionary.csv` documents source names, feature origins and definitions. `quality_report.json` records the source hash, settings, missing counts, checks and split dates. Rerunning replaces these generated files in OUTPUT_DIR; the original Excel is untouched.


In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
inverse_mapping = {v: k for k, v in column_mapping.items()}
dictionary = [
    {"column": "date", "role": "metadata", "source_column": "Date", "origin": "source", "lag_sessions": 0,
     "definition": "Target session date; do not include in X"},
    {"column": "feature_date", "role": "metadata", "source_column": "Date", "origin": "shifted", "lag_sessions": 1,
     "definition": "Previous observed session; latest session used by predictors"},
    {"column": "split", "role": "metadata", "source_column": "", "origin": "chronological_split", "lag_sessions": 0,
     "definition": "train / validation / test; fixed common split"},
]
for name in features_at_close.columns:
    dictionary.append({"column": name+"_lag1", "role": "feature",
                       "source_column": inverse_mapping.get(name, ""), "origin": feature_origins[name],
                       "lag_sessions": 1, "definition": feature_definitions[name] + "; evaluated at feature_date"})
dictionary.append({"column": TARGET, "role": "target", "source_column": "Daily_move", "origin": "validated_source",
                   "lag_sessions": 0, "definition": "1 if close at date > close at previous observed session; otherwise 0"})
schema = {"schema_version": SCHEMA_VERSION, "target_column": TARGET,
          "metadata_columns": ["date", "feature_date", "split"], "feature_columns": FEATURE_COLUMNS,
          "feature_lag_sessions": 1, "prediction_convention": "Previous observed session close to target session close",
          "split_order": ["train", "validation", "test"]}
report = {
    "schema_version": SCHEMA_VERSION,
    "source_file": INPUT_PATH.name,
    "source_sha256": hashlib.sha256(INPUT_PATH.read_bytes()).hexdigest(),
    "source_rows": len(raw), "source_columns": len(raw.columns),
    "validated_rows": len(clean), "model_rows": len(model_data), "feature_count": len(FEATURE_COLUMNS),
    "source_technical_indicator_count": len(raw.columns)-7,
    "added_feature_count": sum(v == "derived" for v in feature_origins.values()),
    "missing_counts_before_feature_engineering": {k: int(v) for k,v in input_missing.items() if v},
    "excluded_rows": len(excluded_rows), "interior_exclusions": int(interior_exclusions),
    "missing_values_in_final_table": int(model_data.isna().sum().sum()),
    "split_fractions": {"train": TRAIN_FRACTION, "validation": VALID_FRACTION, "test": 1-TRAIN_FRACTION-VALID_FRACTION},
    "splits": split_summary, "moving_average_checks": indicator_checks,
    "cleaning_log": cleaning_log,
    "policies": {"lag_sessions": 1, "imputation": "none; exclude incomplete model rows after lagging",
                 "outliers": "retain valid market extremes", "scaling": "not performed; fit on training data only",
                 "external_data": False, "supplied_indicators": "retain source definitions except recomputed moving averages"},
    "validation": "Passed finite-value, label, date, split, one-session alignment and prefix checks",
    "versions": {"python": platform.python_version(), "pandas": pd.__version__, "numpy": np.__version__},
}
model_data.to_csv(OUTPUT_DIR/"sp500_model_ready.csv", index=False, date_format="%Y-%m-%d")
pd.DataFrame(dictionary).to_csv(OUTPUT_DIR/"data_dictionary.csv", index=False)
excluded_rows.to_csv(OUTPUT_DIR/"excluded_rows.csv", index=False, date_format="%Y-%m-%d")
pd.DataFrame(cleaning_log).to_csv(OUTPUT_DIR/"cleaning_log.csv", index=False)
for name, content in [("feature_columns.json", schema), ("quality_report.json", report)]:
    (OUTPUT_DIR/name).write_text(json.dumps(content, indent=2, ensure_ascii=False, allow_nan=False), encoding="utf-8")
print("Saved:", ", ".join(sorted(p.name for p in OUTPUT_DIR.iterdir() if p.is_file())))


Saved: cleaning_log.csv, data_dictionary.csv, excluded_rows.csv, feature_columns.json, quality_report.json, sp500_model_ready.csv


## 9. Group-member loading example
Copy this cell into each model notebook and set `DATA_DIR`. It returns pandas DataFrames with identical named feature columns, and integer target Series. Fit any scaler/selector inside a training pipeline. Keep the held-out test data out of model selection. Do not run another `shift(1)` or a new random split.


In [ ]:
DATA_DIR = OUTPUT_DIR  # In another notebook: Path("/path/to/processed")
schema = json.loads((DATA_DIR/"feature_columns.json").read_text(encoding="utf-8"))
dataset = pd.read_csv(DATA_DIR/"sp500_model_ready.csv", parse_dates=["date", "feature_date"])
FEATURE_COLUMNS = schema["feature_columns"]
TARGET = schema["target_column"]

def get_split(name):
    rows = dataset.loc[dataset["split"].eq(name)].copy()
    X = rows.loc[:, FEATURE_COLUMNS].copy()
    y = rows[TARGET].astype("int8").copy()
    return X, y

X_train, y_train = get_split("train")
X_valid, y_valid = get_split("validation")
X_test, y_test = get_split("test")
assert list(X_train.columns) == list(X_valid.columns) == list(X_test.columns)
assert np.isfinite(dataset[FEATURE_COLUMNS].to_numpy()).all()
np.testing.assert_allclose(dataset[FEATURE_COLUMNS].to_numpy(), model_data[FEATURE_COLUMNS].to_numpy(), rtol=1e-12, atol=1e-12)
print("Train:", X_train.shape, "Validation:", X_valid.shape, "Test:", X_test.shape)
print("Target:", TARGET, "| classes: 0 = down/flat, 1 = up")


Train: (6294, 153) Validation: (1348, 153) Test: (1350, 153)
Target: daily_move | classes: 0 = down/flat, 1 = up
